# IND320 Part 4 — Machine learning: forecasts, exchange rates and a refactored app

This notebook continues Parts 1–3. Part 4 adds exchange rates from **Norges Bank**, investigates **time zones**
in the four data sources, builds a **weekly weather history** for every reservoir area, studies the **sliding window
correlation** between weather and reservoirs, and develops **forecasts of the fill level** with the River framework.
The Streamlit app is refactored at the same time (Plotly charts, grouped menu, map, new pages).

**Contents**

1. Setup
2. Sketch of the new app structure
3. Exchange rates from Norges Bank
4. Time zones in NVE, ENTSO-E, Open-Meteo and Norges Bank
5. Weekly weather history for all areas (Open-Meteo → MongoDB)
6. Sliding window correlation: weather against reservoirs
7. Forecasting the fill level with River (SNARIMAX and lag regression)
8. AI usage
9. Work log

## Project links

- Public GitHub repository: https://github.com/yaksh-git/ind320-dashboard-basics-yaksh-git
  (Part 4 is developed on the branch `ind320-part4` and merged into `main` after peer review 3 and feedback 3)
- Streamlit app: https://ind320-dashboard-basics-yaksh-git.streamlit.app/
- Screencast (Part 4): https://github.com/yaksh-git/ind320-dashboard-basics-yaksh-git/blob/main/docs/part_4/screencast_part4.mp4
- The AI-usage statement and the work log are in sections 8 and 9 at the end.

## 1. Setup

The notebook lives in `notebooks/` and uses the project's Python 3.12 environment (`notebooks/requirements.txt`).
Following the advice from peer review 3, the first code cell contains **only the imports**; the second cell sets up
paths, the plot style and the connection settings. Secrets (the MongoDB connection string) are read from the
Git-ignored file `.streamlit/secrets.toml`; only the section names are printed, never the values.

In [ ]:
# Imports only (peer review 3: imports in the first cell, setup in the next one)
import inspect
import json
import sys
import time
import tomllib
from datetime import date
from io import StringIO
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import Image, display
from pymongo import ASCENDING, MongoClient, UpdateOne
from river import linear_model, optim, preprocessing, time_series

In [ ]:
# Setup: paths, plot style, secrets, URLs and the app's shared forecasting code (src/utils/forecast.py)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()  # Jupyter starts in notebooks/
REGION_DIR = ROOT / 'data' / 'regions'            # small map files that the app also uses (in Git)
CACHE_DIR = ROOT / 'data' / 'cache'               # large downloads (Git-ignored, re-created by this notebook)
DAILY_CACHE = CACHE_DIR / 'weather_daily'
DAILY_CACHE.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='notebook', palette='colorblind')

with open(ROOT / '.streamlit' / 'secrets.toml', 'rb') as file:
    secrets = tomllib.load(file)
print('Secret sections found:', ', '.join(secrets))  # names only, never the values

NVE_URL = 'https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData'
ARCHIVE_URL = 'https://archive-api.open-meteo.com/v1/archive'
NB_URL = 'https://data.norges-bank.no/api/data/EXR/B.{currencies}.NOK.SP'

# The forecasting function is shared with the app, so the notebook tests exactly the code the app runs.
sys.path.insert(0, str(ROOT))
from src.utils import forecast as fc  # noqa: E402 (needs the path above)
print('pandas', pd.__version__, '| River model classes:', time_series.SNARIMAX.__name__)

## 2. Sketch of the new app structure

Before changing the code I drew the new structure on paper (the assignment asks for a sketch before refactoring).
The sketch shows an A4 page laid out like the app: the menu with its groups on the left, and the new **Map** page on
the right. The circles mark the three new pages in Part 4.

In [ ]:
# Show the hand-drawn sketch (photo, cropped to the paper)
display(Image(filename=str(ROOT / 'docs' / 'part_4' / 'sketch_app_structure.jpg'), width=620))

**Decisions shown in the sketch**

- **Grouping by type of task.** Parts 2–3 grouped the menu by data source. The new pages combine sources (the forecast
  uses NVE, Open-Meteo and Norges Bank), so the menu is now grouped by task: **Start** (Home, Map), **Explore**
  (Reservoir data, Fill-level analysis, Weather vs. reservoir), **Anomalies** (Weather outliers), **Forecast**
  (Reservoir forecast), **Electricity** (Cross-border flows) and **Info** (About). Each group has its own colour, used in the
  page header and on the active menu entry, so the user can see which group is active. The Home page has one card per
  group that leads to its main page.
- **The LineChartColumn page.** The old *Data table* page (with `LineChartColumn`) and *Data plot* page showed the same
  data with the same controls, so they are joined into one page, **Reservoir data**, with the tabs *Table* and *Plot*.
- **Global and local selections.** The area and the time period are global: a click on the map (thick outline) stores
  the area in `st.session_state`, and the analysis pages open with it; the map's time period is also the training
  period of the forecast. Choices that only make sense on one page stay local (year on Weather outliers, lag and window
  on Weather vs. reservoir, model parameters on Forecast).

## 3. Exchange rates from Norges Bank

Norges Bank publishes its exchange rates through an open SDMX REST API (no key). The data set `EXR` is addressed by
a key with four parts: frequency `B` (business days), base currency (`EUR+USD`), quote currency `NOK` and the type
`SP` (spot). With `format=csv` the answer is a semicolon-separated table with one row per currency and day.
The app uses the same function (`src/utils/currency.py`), with caching, for the exchange rate as an exogenous
variable in the forecast.

In [ ]:
# Function: download daily EUR/NOK and USD/NOK between two dates from Norges Bank (one column per currency)
def download_exchange_rates(start: str, end: str, currencies: tuple[str, ...] = ('EUR', 'USD')) -> pd.DataFrame:
    """Daily rates in NOK per unit of each currency, indexed by date (business days only)."""
    url = NB_URL.format(currencies='+'.join(currencies))
    params = {'format': 'csv', 'startPeriod': start, 'endPeriod': end, 'locale': 'en'}
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()  # stop with a clear error on HTTP 4xx/5xx
    rows = pd.read_csv(StringIO(response.text), sep=';')
    rows['TIME_PERIOD'] = pd.to_datetime(rows['TIME_PERIOD'], format='%Y-%m-%d')
    # From long (one row per currency and day) to wide (one column per currency).
    rates = rows.pivot(index='TIME_PERIOD', columns='BASE_CUR', values='OBS_VALUE')
    return rates.rename_axis(index='date', columns=None)

In [ ]:
# Test: download from 2016 to today and check that the result looks like exchange rates
rates = download_exchange_rates('2016-01-01', f'{date.today():%Y-%m-%d}')
assert list(rates.columns) == ['EUR', 'USD'], 'one column per currency'
assert rates.index.dayofweek.max() <= 4, 'business days only (Monday = 0, Friday = 4)'
assert rates.notna().all().all(), 'no missing values'
assert rates['EUR'].between(8, 14).all() and rates['USD'].between(6, 13).all(), 'plausible NOK rates'
print(f'{len(rates):,} business days from {rates.index[0]:%d %b %Y} to {rates.index[-1]:%d %b %Y}')
display(rates.tail(3))
display(rates.describe().round(3))

In [ ]:
# Plot: daily rates and weekly means (both in NOK per unit of currency, so one y-axis is correct)
weekly_rates = rates.resample('W-SUN').mean()  # weeks that end on Sunday, like NVE's weeks
fig, ax = plt.subplots(figsize=(12, 4))
for currency, colour in [('EUR', '#0173B2'), ('USD', '#DE8F05')]:
    ax.plot(rates.index, rates[currency], color=colour, linewidth=0.5, alpha=0.45, label=f'{currency}/NOK daily')
    ax.plot(weekly_rates.index, weekly_rates[currency], color=colour, linewidth=1.4, label=f'{currency}/NOK weekly mean')
ax.set(title='Exchange rates from Norges Bank', xlabel='', ylabel='NOK per unit of currency')
ax.legend(ncol=2, frameon=False)
plt.show()

**Findings.** TBD

## 4. Time zones in NVE, ENTSO-E, Open-Meteo and Norges Bank

The four sources describe time in different ways. The checks below look at the raw data of each source; the summary
table at the end gives the solution used in the notebook and the app.

In [ ]:
# NVE: dato_Id is a plain date without time or time zone; check its weekday and the ISO week fields
nve = pd.DataFrame(requests.get(NVE_URL, headers={'Accept': 'application/json'}, timeout=60).json())
nve_dates = pd.to_datetime(nve['dato_Id'], format='%Y-%m-%d')
nve['date'] = nve_dates  # used again in sections 6 and 7
iso = nve_dates.dt.isocalendar()
print('Raw values (newest row): dato_Id =', nve['dato_Id'].iloc[-1],
      '| neste_Publiseringsdato =', nve['neste_Publiseringsdato'].iloc[-1])
print('Weekdays of dato_Id:', nve_dates.dt.day_name().value_counts().to_dict())
assert (iso['week'].to_numpy() == nve['iso_uke'].to_numpy()).all(), 'iso_uke is the ISO week of dato_Id'
assert (iso['year'].to_numpy() == nve['iso_aar'].to_numpy()).all(), 'iso_aar is the ISO year of dato_Id'
print('iso_aar and iso_uke agree with dato_Id in all', f'{len(nve):,}', 'rows')

In [ ]:
# ENTSO-E: the Part 2 cache keeps the API's UTC times; count 15-minute values per Norwegian day around the DST changes
flows = pd.read_parquet(CACHE_DIR / 'entsoe_physical_flows_2024_2025.parquet', columns=['time', 'link', 'direction'])
nl = flows.loc[(flows['link'] == 'NO2-DE') & (flows['direction'] == 'import'), 'time']  # NO2-DE: 15-minute values only
print('Stored time zone:', nl.dt.tz, '| first value:', nl.min(), '=', nl.min().tz_convert('Europe/Oslo'), 'in Oslo')
local_days = nl.dt.tz_convert('Europe/Oslo').dt.date
per_day = local_days.value_counts()
for day in [date(2024, 3, 30), date(2024, 3, 31), date(2024, 10, 27)]:
    print(f'{day}: {per_day[day]} quarter-hours ({per_day[day] / 4:g} hours) in Norwegian local time')

In [ ]:
# Open-Meteo: the same location and days with the default timezone (GMT) and with timezone=Europe/Oslo
answers = {}
for zone in ['GMT', 'Europe/Oslo']:
    params = {'latitude': 63.5, 'longitude': 10.5, 'start_date': '2024-01-10', 'end_date': '2024-01-11',
              'daily': 'precipitation_sum', 'hourly': 'temperature_2m', 'models': 'era5', 'timezone': zone}
    answers[zone] = requests.get(ARCHIVE_URL, params=params, timeout=60).json()
for zone, answer in answers.items():
    print(f"{zone:12} utc_offset_seconds={answer['utc_offset_seconds']:5} first hour='{answer['hourly']['time'][0]}' "
          f"daily precipitation={answer['daily']['precipitation_sum']} mm")

In [ ]:
# Norges Bank: rates exist only on business days, and the raw CSV says when they are fixed
raw = pd.read_csv(StringIO(requests.get(NB_URL.format(currencies='EUR'), timeout=60,
                  params={'format': 'csv', 'startPeriod': '2025-12-20', 'endPeriod': '2026-01-05',
                          'locale': 'en'}).text), sep=';')
print('Collection indicator:', raw['Collection Indicator'].iloc[0])
print('Dates around Christmas 2025:', ', '.join(raw['TIME_PERIOD']))
print('Weekdays in the whole download:', rates.index.day_name().value_counts().to_dict())

**Findings and solution**

| Source | How time is given | Problem | Solution used |
|---|---|---|---|
| NVE (Magasinstatistikk) | `dato_Id` is a date without time (always a **Sunday**, the end of the ISO week); `iso_aar`/`iso_uke` agree with it; `neste_Publiseringsdato` has a time but no zone (Norwegian local time) | none for weekly data, but other data must be put on the same weeks | All weekly series use weeks that end on Sunday (`resample('W-SUN')`), labelled by that Sunday |
| ENTSO-E | UTC timestamps (`…Z`), 15- or 60-minute resolution | a Norwegian day has 23 or 25 hours at the daylight-saving changes; a UTC day ends at 01:00/02:00 in Norway | Store in UTC; convert to `Europe/Oslo` before grouping by day, month or year (done in Part 2) |
| Open-Meteo | `timezone` parameter, **GMT by default**; times are local strings without an offset; `utc_offset_seconds` gives the offset | daily sums depend on the zone (5.8 mm in Oslo time vs 6.3 mm in GMT above) | Hourly data in UTC (Part 3); daily data with `timezone=Europe/Oslo`, so a day is a Norwegian day |
| Norges Bank | dates only, **business days only**, fixed at about 14:15 CET (ECB concertation) | no values on weekends and holidays; the fixing time is one moment of the day | Weekly mean of the business days in each NVE week; weeks without a rate are filled with the previous week |

After these steps every series is on the same weekly grid as NVE: one value per ISO week, labelled by the Sunday that
ends it, with days counted in Norwegian time. The remaining difference is that ENTSO-E and hourly weather are measured
in UTC hours, but this does not matter after summing to weeks (at most one or two hours move between neighbouring weeks).

## 5. Weekly weather history for all areas (Open-Meteo → MongoDB)

The map, the correlation page and the forecast need **many years of weather for every area**. Part 3 downloaded one
year of hourly data at 20 locations when the page needed it; that does not scale to ten years and eight areas.
Open-Meteo's free plan allows 10,000 calls per day, 5,000 per hour and 600 per minute, and a request counts as one
call per location for each started 14 days (with at most 10 variables). Ten years at one location therefore cost
about 280 calls.

The design is therefore:

- **6 locations per area**, drawn from the Part 3 locations with at most one per ERA5 grid cell (0.25°). ERA5 is smooth
  at this scale, and section 5.4 checks the weekly values against the 20 locations from Part 3.
- **Daily** values (mean temperature, precipitation sum, snowfall sum) with `timezone=Europe/Oslo`, from 2016 onwards.
- Each area–year is **cached** as a Parquet file and the downloads are **throttled** to 4,500 calls per hour. The 8 areas
  cost about 13,600 calls, so the download was spread over two days (price areas on day 1, watercourse regions on day 2).
- The daily values are averaged over the locations and aggregated to **ISO weeks that end on Sunday** (as NVE).
- **Norway (NO 0)** is the land-area-weighted mean of the five price areas, which together cover the country.
- The weekly table (about 5,000 rows) is stored in **MongoDB**; the app reads it from there.

### 5.1 Locations (from CSV to GeoJSON)

Peer review 3 pointed out the CSV file with the random locations from Part 3. The locations are now stored as GeoJSON
(`data/regions/sample_points.geojson`), with one `MultiPoint` per area, which is also the map format the assignment asks
for. The CSV is no longer in the repository; the cell converts it once if it is still on disk.

In [ ]:
# Random locations from Part 3: convert the CSV to GeoJSON once, then read the GeoJSON (one MultiPoint per area)
points_csv, points_file = REGION_DIR / 'sample_points.csv', REGION_DIR / 'sample_points.geojson'
if not points_file.exists():
    old = pd.read_csv(points_csv)
    features = [{'type': 'Feature', 'properties': {'area_type': kind, 'area_number': int(number)},
                 'geometry': {'type': 'MultiPoint', 'coordinates': group[['lon', 'lat']].round(4).values.tolist()}}
                for (kind, number), group in old.groupby(['area_type', 'area_number'], sort=False)]
    points_file.write_text(json.dumps({'type': 'FeatureCollection', 'features': features}, separators=(',', ':')))

point_features = json.loads(points_file.read_text(encoding='utf-8'))['features']
points = pd.DataFrame([{'area_type': f['properties']['area_type'], 'area_number': f['properties']['area_number'],
                        'lon': lon, 'lat': lat}
                       for f in point_features for lon, lat in f['geometry']['coordinates']])
print(f'{points_file.name}: {points_file.stat().st_size / 1000:.0f} kB,', len(points), 'points')
print(points.groupby(['area_type', 'area_number']).size().to_dict())

### 5.2 Download daily weather per area

In [ ]:
# Functions: choose one location per ERA5 cell, download daily ERA5 values, and cache each area-year as Parquet
DAILY_VARIABLES = ['temperature_2m_mean', 'precipitation_sum', 'snowfall_sum']
N_LOCATIONS, SEED, ERA5_GRID = 6, 320, 0.25
FIRST_DAY, LAST_DAY = pd.Timestamp('2016-01-01'), pd.Timestamp('2026-11-08')  # last NVE week when this was written
CALLS_PER_HOUR = 4500  # below the free limit of 5,000 calls per hour


def choose_locations(points: pd.DataFrame, area_type: str, area_number: int) -> pd.DataFrame:
    """N_LOCATIONS random locations of one area, at most one per ERA5 grid cell (lat, lon on the grid)."""
    rows = points.loc[(points['area_type'] == area_type) & (points['area_number'] == area_number)]
    # Round to the 0.25° grid: two points in the same cell would return identical weather.
    cells = rows.assign(lat=(rows['lat'] / ERA5_GRID).round() * ERA5_GRID,
                        lon=(rows['lon'] / ERA5_GRID).round() * ERA5_GRID)
    return cells.drop_duplicates(['lat', 'lon']).sample(N_LOCATIONS, random_state=SEED)[['lat', 'lon']]


def download_daily(locations: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    """One Open-Meteo request for all locations; daily values in Norwegian days, one row per location and day."""
    params = {'latitude': ','.join(map(str, locations['lat'])), 'longitude': ','.join(map(str, locations['lon'])),
              'start_date': f'{start:%Y-%m-%d}', 'end_date': f'{end:%Y-%m-%d}', 'daily': ','.join(DAILY_VARIABLES),
              'models': 'era5', 'timezone': 'Europe/Oslo'}
    while True:
        response = requests.get(ARCHIVE_URL, params=params, timeout=300)
        if response.status_code != 429:
            break
        print('Rate limit:', response.json().get('reason'), '- waiting 15 minutes')  # HTTP 429 = too many calls
        time.sleep(15 * 60)
    response.raise_for_status()
    answer = response.json()
    answer = answer if isinstance(answer, list) else [answer]  # one location gives one object, not a list
    return pd.concat([pd.DataFrame(item['daily']).assign(location=i) for i, item in enumerate(answer)],
                     ignore_index=True)


def area_daily_weather(points: pd.DataFrame, area_type: str, area_number: int) -> pd.DataFrame:
    """Daily weather of one area (mean of its locations), downloaded year by year or read from the cache."""
    locations = choose_locations(points, area_type, area_number)
    frames = []
    for year in range(FIRST_DAY.year, LAST_DAY.year + 1):
        target = DAILY_CACHE / f'{area_type}{area_number}_{year}.parquet'
        if not target.exists():
            start, end = max(FIRST_DAY, pd.Timestamp(f'{year}-01-01')), min(LAST_DAY, pd.Timestamp(f'{year}-12-31'))
            download_daily(locations, start, end).to_parquet(target)
            calls = N_LOCATIONS * ((end - start).days + 1) / 14
            time.sleep(calls / CALLS_PER_HOUR * 3600)  # stay below the hourly limit
        frames.append(pd.read_parquet(target))
    daily = pd.concat(frames)
    daily['time'] = pd.to_datetime(daily['time'], format='%Y-%m-%d')
    return daily.groupby('time')[DAILY_VARIABLES].mean()  # area mean per day

### 5.3 From days to weeks

In [ ]:
# Function: weekly weather from daily values (weeks end on Sunday like NVE; incomplete weeks are dropped)
def weekly_weather(daily: pd.DataFrame) -> pd.DataFrame:
    """Weekly mean temperature (°C), precipitation (mm per week) and snowfall (cm per week)."""
    weeks = daily.resample('W-SUN')
    weekly = pd.DataFrame({
        'temperature': weeks['temperature_2m_mean'].mean(),
        # min_count=7: a week with a missing day gets no sum instead of a too small one.
        'precipitation': weeks['precipitation_sum'].sum(min_count=7),
        'snowfall': weeks['snowfall_sum'].sum(min_count=7),
    })
    return weekly.dropna()


AREAS = [('EL', n) for n in range(1, 6)] + [('VASS', n) for n in range(1, 4)]
weekly_by_area = {(kind, number): weekly_weather(area_daily_weather(points, kind, number)) for kind, number in AREAS}

# Norway (NO 0): the price areas cover the whole country, so their land-area-weighted mean is the national value.
area_features = json.loads((REGION_DIR / 'areas.geojson').read_text(encoding='utf-8'))['features']
land_km2 = {(f['properties']['area_type'], f['properties']['area_number']): f['properties']['land_km2']
            for f in area_features}
weights = {n: land_km2[('EL', n)] for n in range(1, 6)}
weekly_by_area[('NO', 0)] = sum(weekly_by_area[('EL', n)] * w for n, w in weights.items()) / sum(weights.values())

weather_weekly = pd.concat({key: frame for key, frame in weekly_by_area.items()},
                           names=['area_type', 'area_number', 'week']).reset_index()
assert weather_weekly.groupby(['area_type', 'area_number']).size().nunique() == 1, 'same weeks for every area'
print(f"{len(weather_weekly):,} rows: {weather_weekly['week'].min():%d %b %Y} – {weather_weekly['week'].max():%d %b %Y}")
summary = weather_weekly.groupby(['area_type', 'area_number']).agg(
    mean_temperature_C=('temperature', 'mean'), precipitation_mm_per_year=('precipitation', lambda s: s.mean() * 52.18),
    snowfall_cm_per_year=('snowfall', lambda s: s.mean() * 52.18))
display(summary.round(1))

### 5.4 Check: 6 locations against the 20 locations of Part 3

Part 3 downloaded hourly weather for NO 3 in 2024 at 20 locations (cached in `data/cache`). If 6 locations are
enough for weekly values, the two weekly series should be almost identical.

In [ ]:
# Validation: weekly NO 3 (EL 3) weather from 6 daily locations vs 20 hourly locations from Part 3 (year 2024)
hourly = pd.read_parquet(CACHE_DIR / 'weather_NO3_2024.parquet')
hourly_mean = hourly.groupby('time')[['temperature_2m', 'precipitation']].mean()
hourly_mean.index = hourly_mean.index.tz_convert('Europe/Oslo').tz_localize(None)  # Norwegian days, as the daily data
part3 = pd.DataFrame({'temperature': hourly_mean['temperature_2m'].resample('W-SUN').mean(),
                      'precipitation': hourly_mean['precipitation'].resample('W-SUN').sum()})
both = weekly_by_area[('EL', 3)].join(part3, rsuffix='_part3', how='inner').iloc[1:-1]  # complete weeks of 2024
for column in ['temperature', 'precipitation']:
    r = both[column].corr(both[f'{column}_part3'])
    difference = (both[column] - both[f'{column}_part3']).abs().mean()
    print(f'{column:13}: correlation {r:.3f}, mean absolute difference {difference:.2f} ({len(both)} weeks)')
    assert r > 0.95, 'the 6-location weekly series should follow the 20-location series closely'

### 5.5 Store the weekly weather in MongoDB

In [ ]:
# Upload: one document per area and week; a unique index and upserts make it safe to run the cell again
client = MongoClient(secrets['mongo']['uri'], serverSelectionTimeoutMS=10_000)
collection = client[secrets['mongo'].get('database', 'ind320')]['weather_weekly']
collection.create_index([('area_type', ASCENDING), ('area_number', ASCENDING), ('week', ASCENDING)], unique=True)
documents = weather_weekly.assign(area_number=weather_weekly['area_number'].astype(int)).round(3).to_dict('records')
result = collection.bulk_write([UpdateOne({k: doc[k] for k in ('area_type', 'area_number', 'week')},
                                          {'$set': doc}, upsert=True) for doc in documents])
print(f'inserted {result.upserted_count:,}, updated {result.modified_count:,}, unchanged '
      f'{len(documents) - result.upserted_count - result.modified_count:,}')
stored = collection.count_documents({})
assert stored == len(weather_weekly), 'MongoDB holds exactly one document per area and week'
print(f'{stored:,} documents in {collection.full_name}')

**Findings.** TBD

## 6. Sliding window correlation: weather against reservoirs

The sliding window correlation (SWC) computes the correlation between two series in a window that moves along the time
axis. The weather series is shifted by a **lag**: with a positive lag, the reservoir in week *t* is compared with the
weather in week *t − lag*, so the weather comes first. As the representative reservoir variable I use the **weekly change
in fill level** (in percentage points), because inflow and outflow show up in the change, not in the level itself.
The same functions are used on the app page *Weather vs. reservoir* (`src/utils/analysis.py`).

In [ ]:
# Functions: sliding window correlation at one lag, and the correlation over the whole period for every lag
def sliding_correlation(weather: pd.Series, reservoir: pd.Series, lag: int, window: int) -> pd.Series:
    """Correlation in a window of `window` weeks between the weather lag weeks earlier and the reservoir now."""
    pairs = pd.concat([weather.shift(lag), reservoir], axis=1).dropna()
    return pairs.iloc[:, 0].rolling(window, min_periods=int(0.8 * window)).corr(pairs.iloc[:, 1])


def lag_correlations(weather: pd.Series, reservoir: pd.Series, max_lag: int = 26) -> pd.Series:
    """Correlation over the whole period for each lag from -max_lag to +max_lag weeks."""
    return pd.Series({lag: weather.shift(lag).corr(reservoir) for lag in range(-max_lag, max_lag + 1)})


# Weekly change in fill level per area (percentage points), from the NVE data downloaded in section 4.
change = {(kind, int(number)): group.set_index('date')['endring_fyllingsgrad'].sort_index() * 100
          for (kind, number), group in nve.groupby(['omrType', 'omrnr'])}

In [ ]:
# Example: NO 3, snowfall against the weekly change, at lag 0 and at the lag with the strongest correlation
area, window = ('EL', 3), 26
snow = weekly_by_area[area]['snowfall']
reservoir = change[area].loc[snow.index[0]:snow.index[-1]]
by_lag = lag_correlations(snow, reservoir)
best_lag = int(by_lag.abs().idxmax())
print(f'NO 3: correlation at lag 0 = {by_lag[0]:.2f}; strongest at lag {best_lag} weeks = {by_lag[best_lag]:.2f}')

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(snow.index, snow, color='#0173B2', linewidth=0.8)
axes[0].set(ylabel='Snowfall\n(cm per week)', title=f'NO 3: {window}-week sliding correlation, snowfall vs change in fill level')
axes[1].plot(reservoir.index, reservoir, color='#087F8C', linewidth=0.8)
axes[1].set(ylabel='Change in fill\nlevel (pp)')
for lag, colour in [(0, '#949494'), (best_lag, '#B35A12')]:
    axes[2].plot(sliding_correlation(snow, reservoir, lag, window), color=colour, linewidth=1.3, label=f'lag {lag} weeks')
axes[2].axhline(0, color='#12263A', linewidth=0.6)
axes[2].set(ylabel='Correlation', ylim=(-1, 1))
axes[2].legend(frameon=False, ncol=2)
plt.show()

In [ ]:
# Lag scan: strongest lag and correlation for every area and weather property (whole period, change in fill level)
rows = []
for area, frame in weekly_by_area.items():
    reservoir = change[area].loc[frame.index[0]:frame.index[-1]]
    for prop in ['temperature', 'precipitation', 'snowfall']:
        by_lag = lag_correlations(frame[prop], reservoir)
        best = int(by_lag.abs().idxmax())
        rows.append({'area': f'{area[0]} {area[1]}', 'property': prop, 'lag 0': by_lag[0],
                     'strongest lag (weeks)': best, 'correlation there': by_lag[best]})
lag_table = pd.DataFrame(rows).pivot(index='area', columns='property',
                                     values=['lag 0', 'strongest lag (weeks)', 'correlation there'])
display(lag_table.round(2))

**Findings.** TBD

## 7. Forecasting the fill level with River

**River** is a framework for online machine learning: a model learns from one observation at a time (`learn_one`)
and can keep learning when new data arrive. Two models are compared:

- **SNARIMAX** (`river.time_series.SNARIMAX`): seasonal ARIMA with exogenous variables. The orders *p, d, q* describe
  the recent weeks, the seasonal orders *sp, sd, sq* the same weeks in earlier years (season *m* = 52 weeks). The
  regression behind it is fitted online with stochastic gradient descent (SGD).
- **Lag regression**: a River linear regression (also SGD) that predicts **next week's change** in fill level from the
  current level, the last *p* changes, the change in the same week last year (*sp* = 1), the time of year (sine and
  cosine terms) and the chosen exogenous variables. The fill level is the running sum of the changes.

Both forecasts are **dynamic**: each forecast week is built on the forecasts of the weeks before it, not on observed
values. The future weather is unknown, so the forecast uses the **weekly climatology** (the mean of the same ISO week in
the training years); the exchange rate keeps its last value. For the **confidence band** (bonus task) two extra models
are trained with `optim.losses.Quantile` (the pinball loss), for example for the 10 % and 90 % quantiles.

The function is in `src/utils/forecast.py`, so the app runs exactly the code tested here. Its inputs are the series,
the exogenous variables, the end of the training period and a `Settings` object with the model type, the orders, the
history length, the horizon, the learning rate, the number of epochs and the band width.

### 7.1 The forecasting function

In [ ]:
# Show the main parts of the shared forecasting code: settings, lag features, future exogenous values and the function
for function in [fc.Settings, fc.lag_features, fc.future_exogenous, fc.forecast_fill]:
    print(inspect.getsource(function))

### 7.2 Time resolution

NVE publishes one value per week, so the natural resolution is weekly. To check whether a finer or coarser resolution
would be better, the same SNARIMAX model (p = 1, sd = 1, five years of history, half a year ahead) is run on **daily**
values (the weekly fill level interpolated to days, season 365), **weekly** values (season 52) and **4-weekly** means
(season 13), from 10 forecast origins for Norway (NO 0). SGD needs a learning rate that suits the number of values, so
each resolution is run with three learning rates.

In [ ]:
# Time resolution test: SNARIMAX on daily, weekly and 4-weekly fill level (NO 0, 10 origins, 5 years, half a year ahead)
fill = {(kind, int(number)): (group.set_index('date')['fyllingsgrad'].sort_index() * 100).asfreq('W-SUN').interpolate()
        for (kind, number), group in nve.groupby(['omrType', 'omrnr'])}
RESOLUTIONS = {'daily': ('D', 365, 182), 'weekly': ('W-SUN', 52, 26), '4-weekly': ('4W-SUN', 13, 7)}


def resolution_test(series: pd.Series, rule: str, season: int, horizon: int, origin: pd.Timestamp,
                    rate: float) -> tuple[float, float]:
    """Seconds to train and forecast, and the MAE (pp) of a SNARIMAX model at one resolution."""
    data = series.resample(rule).mean().interpolate() if rule != 'D' else series.resample('D').interpolate()
    train, test = data.loc[:origin].iloc[-5 * season:], data.loc[origin:].iloc[1:horizon + 1]
    model = time_series.SNARIMAX(p=1, d=0, q=0, m=season, sp=0, sd=1, sq=0, regressor=preprocessing.StandardScaler()
                                 | linear_model.LinearRegression(optimizer=optim.SGD(rate), intercept_lr=rate))
    start = time.perf_counter()
    for value in train:
        model.learn_one(value)
    prediction = np.array(model.forecast(horizon=horizon))[:len(test)]
    return time.perf_counter() - start, float(np.mean(np.abs(prediction - test.to_numpy())))


origins = pd.date_range('2021-01-03', '2025-08-31', freq='26W-SUN')
resolution_rows = []
for name, (rule, season, horizon) in RESOLUTIONS.items():
    for rate in [0.0003, 0.001, 0.003]:
        results = np.array([resolution_test(fill[('NO', 0)], rule, season, horizon, origin, rate) for origin in origins])
        resolution_rows.append({'resolution': name, 'learning rate': rate, 'training values': 5 * season,
                                'seconds per forecast': results[:, 0].mean(), 'MAE (pp)': results[:, 1].mean()})
display(pd.DataFrame(resolution_rows).set_index(['resolution', 'learning rate']).round(3))

**Findings.** TBD

### 7.3 Test design and baselines

Each setting is tested on **all 9 areas** (NO 0, EL 1–5, VASS 1–3) from **20 forecast origins** (every 13 weeks from
January 2021 to November 2025, so all seasons are included) with a **26-week horizon**. The error is the mean absolute
error (MAE) in percentage points (pp) over the 26 weeks, averaged over all 180 forecasts. Two simple baselines show what
a model must beat:

- **Persistence**: the fill level stays where it is.
- **Seasonal climatology**: the current level plus the mean change over the same weeks in the last five years.

In [ ]:
# Test setup: areas, origins, the evaluation function and the two baselines
TEST_AREAS = [('NO', 0)] + [('EL', n) for n in range(1, 6)] + [('VASS', n) for n in range(1, 4)]
TEST_ORIGINS = pd.date_range('2021-01-03', '2025-11-02', freq='13W-SUN')
HORIZON = 26
print(len(TEST_AREAS), 'areas ×', len(TEST_ORIGINS), 'origins =', len(TEST_AREAS) * len(TEST_ORIGINS), 'forecasts per setting')


def evaluate(settings: fc.Settings, names: list[str] | None = None) -> dict:
    """Mean MAE (pp) over all test areas and origins, and the time per forecast, for one setting."""
    errors, start = [], time.perf_counter()
    for area in TEST_AREAS:
        exog = fc.build_exogenous(weekly_by_area[area], weekly_rates, names) if names else None
        for origin in TEST_ORIGINS:
            result = fc.forecast_fill(fill[area], exog, origin, settings, with_band=False)
            errors.append(fc.mean_absolute_error(result, fill[area]))
    return {'MAE (pp)': np.mean(errors), 'median MAE': np.median(errors),
            'seconds per forecast': (time.perf_counter() - start) / len(errors)}


def baseline_errors(area: tuple, origin: pd.Timestamp) -> tuple[float, float]:
    """MAE of persistence and of the seasonal climatology baseline for one area and origin."""
    series = fill[area]
    now, actual = series.loc[origin], series.loc[origin:].iloc[1:HORIZON + 1].to_numpy()
    past_changes = []
    for years_back in range(1, 6):  # the same 26 weeks in each of the last five years, as changes from their start
        start = series.index.get_loc(origin) - 52 * years_back
        past_changes.append(series.iloc[start + 1:start + 1 + HORIZON].to_numpy() - series.iloc[start])
    climatology = now + np.mean(past_changes, axis=0)
    return np.mean(np.abs(now - actual)), np.mean(np.abs(climatology - actual))


baselines = np.array([baseline_errors(area, origin) for area in TEST_AREAS for origin in TEST_ORIGINS])
print(f'Persistence: MAE {baselines[:, 0].mean():.2f} pp | Seasonal climatology: MAE {baselines[:, 1].mean():.2f} pp')

### 7.4 Lag regression: history length, learning rate and epochs

First without exogenous variables, with p = 2 recent changes and last year's change (sp = 1).

In [ ]:
# Grid A: history length × learning rate × epochs for the lag regression (no exogenous variables)
grid_a = []
for years in [3, 5, 8]:
    for rate in [0.001, 0.003, 0.01]:
        for epochs in [1, 3, 5]:
            settings = fc.Settings(model='Lag regression', history_years=years, learning_rate=rate, epochs=epochs)
            grid_a.append({'history (years)': years, 'learning rate': rate, 'epochs': epochs, **evaluate(settings)})
grid_a = pd.DataFrame(grid_a).sort_values('MAE (pp)')
display(grid_a.head(8).round(3))
best_a = grid_a.iloc[0]

In [ ]:
# Grid B: number of recent changes (p) and last year's change (sp) with the best values from grid A
grid_b = []
for p in [0, 1, 2, 4]:
    for sp in [0, 1]:
        settings = fc.Settings(model='Lag regression', history_years=int(best_a['history (years)']),
                               learning_rate=best_a['learning rate'], epochs=int(best_a['epochs']), p=p, sp=sp)
        grid_b.append({'p': p, 'sp': sp, **evaluate(settings)})
grid_b = pd.DataFrame(grid_b).sort_values('MAE (pp)')
display(grid_b.round(3))
best_b = grid_b.iloc[0]

### 7.5 SNARIMAX

In [ ]:
# Grid C: a selection of SNARIMAX orders and learning rates (8 years of history, no exogenous variables)
grid_c = []
for p, d, q, sp, sd, sq in [(1, 0, 0, 0, 1, 0), (2, 0, 0, 0, 1, 0), (3, 0, 0, 0, 1, 0), (2, 1, 0, 1, 1, 0),
                            (1, 0, 1, 0, 1, 1), (2, 0, 1, 1, 1, 1)]:
    for rate in [0.001, 0.003]:
        settings = fc.Settings(model='SNARIMAX', history_years=8, p=p, d=d, q=q, sp=sp, sd=sd, sq=sq, learning_rate=rate)
        grid_c.append({'(p,d,q)(sp,sd,sq)': f'({p},{d},{q})({sp},{sd},{sq})', 'learning rate': rate, **evaluate(settings)})
grid_c = pd.DataFrame(grid_c).sort_values('MAE (pp)')
display(grid_c.round(3))

### 7.6 Exogenous variables

The weekly weather starts in 2016, so with exogenous variables the training cannot start earlier. To compare like with
like, every row below uses five years of history (from 2016 for the first origin in 2021).

In [ ]:
# Grid D: exogenous variables with the best lag-regression settings (five years of history in every row)
best_lag = fc.Settings(model='Lag regression', history_years=5, learning_rate=best_a['learning rate'],
                       epochs=int(best_a['epochs']), p=int(best_b['p']), sp=int(best_b['sp']))
exogenous_sets = [[], ['temperature'], ['precipitation'], ['snowfall'], ['snowpack'], ['snowpack', 'temperature'],
                  ['snowpack', 'temperature', 'precipitation'], ['snowpack', 'temperature', 'eur_nok'], ['eur_nok']]
grid_d = pd.DataFrame([{'exogenous variables': ', '.join(names) or '(none)', **evaluate(best_lag, names)}
                       for names in exogenous_sets]).sort_values('MAE (pp)')
display(grid_d.round(3))

In a real forecast the weather of the coming weeks is unknown, and the climatology carries little news. To see how much
the weather *could* help, grid E repeats some rows with the **observed** weather in the forecast weeks (a perfect weather
forecast). This is only possible in a backtest, but it shows the value of the weather information itself.

In [ ]:
# Grid E: climatology (real forecast) against the observed weather in the forecast weeks (perfect weather forecast)
def evaluate_observed(settings: fc.Settings, names: list[str]) -> float:
    """Mean MAE (pp) when the forecast weeks use the observed exogenous values (backtest only)."""
    errors = []
    for area in TEST_AREAS:
        exog = fc.build_exogenous(weekly_by_area[area], weekly_rates, names)
        for origin in TEST_ORIGINS:
            result = fc.forecast_fill(fill[area], exog, origin, settings, with_band=False, observed_future=True)
            errors.append(fc.mean_absolute_error(result, fill[area]))
    return float(np.mean(errors))


grid_e = pd.DataFrame([{'exogenous variables': ', '.join(names),
                        'MAE with climatology (pp)': evaluate(best_lag, names)['MAE (pp)'],
                        'MAE with observed weather (pp)': evaluate_observed(best_lag, names)}
                       for names in [['temperature'], ['precipitation'], ['temperature', 'precipitation'],
                                     ['snowpack', 'temperature', 'precipitation']]])
display(grid_e.round(3))

### 7.7 The default setting and its confidence band

In [ ]:
# Final check of the default setting (the app's default) and the coverage of its 80 % band on all test forecasts
default, default_exog = fc.Settings(), fc.DEFAULT_EXOGENOUS
print(default)
print('Exogenous variables:', default_exog)
errors, inside = [], []
for area in TEST_AREAS:
    exog = fc.build_exogenous(weekly_by_area[area], weekly_rates, default_exog)
    for origin in TEST_ORIGINS:
        result = fc.forecast_fill(fill[area], exog, origin, default, with_band=True)
        actual = fill[area].reindex(result.index)
        errors.append(fc.mean_absolute_error(result, fill[area]))
        inside.append(((actual >= result['lower']) & (actual <= result['upper'])).mean())
print(f'MAE {np.mean(errors):.2f} pp (seasonal climatology {baselines[:, 1].mean():.2f} pp); '
      f'{np.mean(inside):.0%} of the observed weeks are inside the 80 % band')

In [ ]:
# Example: NO 3 forecast through the spring melt (from 2 March 2025) and from the newest week, with the 80 % band
area = ('EL', 3)
exog = fc.build_exogenous(weekly_by_area[area], weekly_rates, fc.DEFAULT_EXOGENOUS)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for axis, origin in zip(axes, [pd.Timestamp('2025-03-02'), exog.index[-1]]):
    result = fc.forecast_fill(fill[area], exog, origin, fc.Settings(), with_band=True)
    history = fill[area].loc[:origin].iloc[-78:]
    axis.plot(history.index, history, color='#087F8C', label='Training data (last 1.5 years)')
    observed = fill[area].reindex(result.index).dropna()
    axis.plot(observed.index, observed, color='#12263A', linestyle=':', label='Observed (not used)')
    axis.plot(result.index, result['forecast'], color='#B35A12', label='Forecast')
    axis.fill_between(result.index, result['lower'], result['upper'], color='#B35A12', alpha=0.15, label='80 % band')
    axis.set(title=f'NO 3, forecast from {origin:%d %b %Y}', ylabel='Fill level (%)', ylim=(0, 100))
    axis.legend(frameon=False, fontsize=9, loc='lower left')
fig.autofmt_xdate()
plt.show()

**Findings and choice of defaults.** TBD

## 8. AI usage

TBD

## 9. Work log

TBD